In [0]:
create or replace temporary view pos as 
with base_table as (
  select *
  from com_edp_prd.cmpa_insights_internal_schema.patient360
),

tx_patients as (
  select distinct
      patient_id,
      coalesce(rendering_npi, referring_npi) as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where ndc11 in ('54092070001','540920700')
    and service_date between '2023-08-01' and '2025-07-31'

  union

  select distinct
      patient_id,
      prescriber_npi as npi,
      fill_date,
      '' as place_of_service
  from com_edp_prd.com_raw.kom_pharmacy_events
  where ndc11 in ('54092070001','540920700')
    and transaction_result = 'PAID'
    and fill_date between '2023-08-01' and '2025-07-31'

  union

  select distinct
      patient_id,
      rendering_npi as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
    and service_date between '2023-08-01' and '2025-07-31'
),

latest_pos_patients as (
  select
      patient_id,
      place_of_service
  from (
    select
        patient_id,
        place_of_service,
        fill_date,
        row_number() over (
          partition by patient_id
          order by fill_date desc
        ) as rn
    from tx_patients
    where place_of_service is not null
      and trim(place_of_service) <> ''
  ) t
  where rn = 1
)

select
    a.*,
    b.place_of_service as latest_place_of_service,
    c.description as pos_description
from base_table a
left join latest_pos_patients b
  on a.patient_id = b.patient_id
left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
  on try_cast(nullif(trim(b.place_of_service), '') as int) = c.code



In [0]:
with tx_patients as (
  select distinct
      patient_id,
      coalesce(rendering_npi, referring_npi) as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where ndc11 in ('54092070001','540920700')
    and service_date between '2023-08-01' and '2025-07-31'

  union

  select distinct
      patient_id,
      prescriber_npi as npi,
      fill_date,
      '' as place_of_service
  from com_edp_prd.com_raw.kom_pharmacy_events
  where ndc11 in ('54092070001','540920700')
    and transaction_result = 'PAID'
    and fill_date between '2023-08-01' and '2025-07-31'

  union

  select distinct
      patient_id,
      rendering_npi as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
    and service_date between '2023-08-01' and '2025-07-31'
)
select * from tx_patients 
-- where patient_id in ('HCZ0G56T','WWH6LCNB','NZGZJ2P1','EYL75Y16','2B8E9JSR')
where patient_id in (select distinct patient_id from pos where latest_place_of_service is null or latest_place_of_service = '')